# Stage 1 - generate (Kaggle GPU T4 x2, Internet ON)
Independent stages; each appends to JSONL under `runs/` and is resumable. **Untested.** Time per stage (my guess, replace after the pilot): original 10-40 min, plans 5-15 min, each steered method 15-50 min for 300 prompts.
Exports `gen_<TAG>_seed<SEED>.zip` (responses = harmful model outputs; keep it OUT of git).

**Pilot timing (measured, 20 prompts, no fast kernels): original 2287 s, plans 163 s, each steered method 1940-2910 s, safe-prompt trio 4300 s. Run notebook 05 first; do not start a 300-prompt run until s per decode step is known.** Optional deviation for speed: add `--max-new-tokens 128` to COMMON (flag it in the blog).

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/sarrazer24/reins-mini"
if not os.path.isdir("/kaggle/working/reins-mini"):
    !git clone {REPO} /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!pip install -q -U transformers accelerate huggingface_hub
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS      # authors' data + frozen controllers; deleted at the end
GUISE = "/tmp/REINS/guise"; CTRL = "/tmp/REINS/configs/qwen35_2b_guise.controllers.json"
from huggingface_hub import snapshot_download
SAE_ROOT = snapshot_download("Carlos4869/REINS-SAE", allow_patterns=["qwen3.5-2b-base/*"], local_dir="/tmp/sae") + "/qwen3.5-2b-base"
print(os.listdir(SAE_ROOT)[:5])

In [ ]:
# Speed kernels for Qwen3.5's linear-attention layers. The pilot ran 114 s per prompt WITHOUT them.
!pip install -q flash-linear-attention
!timeout 1500 pip install -q causal-conv1d --no-build-isolation || echo "causal-conv1d failed or timed out: continue without it"
import importlib
for mod in ("fla", "causal_conv1d"):
    try: importlib.import_module(mod); print(mod, "OK")
    except Exception as e: print(mod, "missing:", type(e).__name__)

In [ ]:
import sys, os, subprocess, glob
TAG, SEED, NPC = "pilot", 0, 4        # pilot: 4 prompts per category = 20 prompts. For the main run: TAG="main", NPC=0 (all 60/category = 300)
OUT = f"runs/{TAG}/seed{SEED}"
COMMON = f"--sae-root {SAE_ROOT} --controllers {CTRL} --guise-dir {GUISE} --out {OUT} --seed {SEED} --n-per-category {NPC}"
def run(cmd): print(">>", cmd); subprocess.run(cmd.replace("python ", "python -u ", 1), shell=True, check=True)

### 1a unsteered control (fp16)

In [ ]:
run(f"python scripts/10_run_stage.py --stage original {COMMON}")

### 1b Harm-Inhibit attribution (fp32 for gradient precision; ~9 GB model + 2.7 GB SAEs)

In [ ]:
run(f"python scripts/10_run_stage.py --stage plans --dtype float32 --bs 1 {COMMON}")

### 1c steered generations

In [ ]:
run(f"python scripts/10_run_stage.py --stage steer --methods reins random_any random_active harm_only refusal_only {COMMON}")

### 1d matched-safe prompts (over-refusal): original, plans, reins

In [ ]:
run(f"python scripts/10_run_stage.py --stage original --field safe {COMMON}")
run(f"python scripts/10_run_stage.py --stage plans --field safe --dtype float32 --bs 1 {COMMON}")
run(f"python scripts/10_run_stage.py --stage steer --field safe --methods reins {COMMON}")

### 1e gate scores

In [ ]:
run(f"python scripts/14_gate_scores.py --sae-root {SAE_ROOT} --controllers {CTRL} --guise-dir {GUISE} --n-per-category {NPC} --out runs/{TAG}/gate_scores_seed{SEED}.csv")

### Export + cleanup

In [ ]:
!cd runs && zip -qr ../gen_{TAG}_seed{SEED}.zip {TAG}
!ls -la gen_*.zip
!rm -rf /tmp/REINS /tmp/sae ~/.cache/huggingface/hub/models--Qwen*
# download gen_*.zip from the notebook Output tab; extract into the repo root (merge folders, never delete earlier results)

In [ ]:
# paste back: sample of 3 responses per method are NOT needed; paste these counts and timings instead
import json
for f in sorted(glob.glob(f"{OUT}/*.jsonl")):
    rows = [json.loads(l) for l in open(f)]
    if rows and "response" in rows[0]:
        n_empty = sum(not r["response"].strip() for r in rows); n_eos = sum(r["eos"] for r in rows)
        print(os.path.basename(f), "n", len(rows), "empty", n_empty, "ended-with-eos", n_eos, "mean chars", sum(len(r["response"]) for r in rows)//max(1,len(rows)))